# 🤖 AI StudyMate — AI-Powered Student Study Assistant

**AI Project Design & Development Lab**

**Pipeline:** Input → AI Logic → Interface → Storage → Action

AI StudyMate lets a student upload study material, ask questions about it, retrieve relevant information using semantic search, generate grounded answers with RAG, create summaries and quizzes, and save the study session.

**Stack:** Python, Google Colab, Gradio, Sentence Transformers, FAISS, PyPDF, Google Gemini API, JSON storage.

In [ ]:
AQ.Ab8RN6LhLCvDNuA_Eaaib62HH1FItE9hf6Cqkc4n9ya2c5BHbw

In [9]:
import getpass
from google import genai

GEMINI_API_KEY = getpass.getpass("Enter your new Gemini API key: ").strip()

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client configured.")

Enter your new Gemini API key: ··········
Gemini client configured.


In [10]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Explain artificial intelligence in one sentence."
)

print(response.text)

Artificial intelligence is the development of computer systems capable of performing tasks that typically require human intelligence, such as learning, reasoning, problem-solving, and decision-making.


In [8]:
models = client.models.list()

for model in models:
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

## 1. Install Dependencies

In [13]:
!pip -q install -U gradio google-genai sentence-transformers faiss-cpu pypdf pandas numpy==1.26.4 scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 39.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... canceled
ERROR: Operation cancelled by user
^C


## 2. Imports

In [1]:
import os
import re
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
import gradio as gr

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google import genai

print("Imports successful.")

Imports successful.


## 3. Gemini API Configuration

Enter your Gemini API key at runtime. It is not stored in the notebook source.

In [2]:
import getpass

GEMINI_API_KEY = getpass.getpass(
    "Enter Gemini API key (leave blank for retrieval-only mode): "
).strip()

if GEMINI_API_KEY:
    client = genai.Client(api_key=GEMINI_API_KEY)
    print("Gemini client configured.")
else:
    client = None
    print("No API key. Retrieval-only mode is available.")

Enter Gemini API key (leave blank for retrieval-only mode): ··········
Gemini client configured.


## 4. Load the Embedding Model

`all-MiniLM-L6-v2` converts document chunks and questions into vector embeddings.

In [3]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded.")
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.
Embedding dimension: 384


/tmp/ipykernel_8084/3426384947.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


## 5. Application State and Storage

The active session is held in memory. Study history is persisted to `study_history.json`.

In [4]:
STATE = {
    "document_name": None,
    "raw_text": "",
    "chunks": [],
    "chunk_metadata": [],
    "embeddings": None,
    "faiss_index": None,
    "chat_history": [],
    "last_answer": "",
    "last_summary": "",
    "last_quiz": "",
}

HISTORY_FILE = Path("study_history.json")

## 6. Document Text Extraction

Supports PDF and TXT. PDF page numbers are retained so retrieved sources can be displayed.

In [5]:
def extract_text_from_file(file_path):
    if not file_path:
        raise ValueError("Please upload a PDF or TXT file.")

    path = Path(file_path)
    suffix = path.suffix.lower()

    if suffix == ".pdf":
        reader = PdfReader(str(path))
        pages = []
        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            pages.append({"page": page_number, "text": text})
        raw_text = "\n".join(p["text"] for p in pages)
    elif suffix == ".txt":
        raw_text = path.read_text(encoding="utf-8", errors="ignore")
        pages = [{"page": 1, "text": raw_text}]
    else:
        raise ValueError("Supported file types: PDF and TXT.")

    if not raw_text.strip():
        raise ValueError("No readable text was found. Scanned/image-only PDFs require OCR.")

    return raw_text, pages


def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

## 7. Text Chunking

Documents are divided into overlapping chunks so retrieval can preserve context across chunk boundaries.

In [6]:
def chunk_pages(pages, chunk_size=900, overlap=150):
    chunks = []
    metadata = []

    for page in pages:
        page_text = clean_text(page["text"])
        if not page_text:
            continue

        start = 0
        while start < len(page_text):
            end = min(start + chunk_size, len(page_text))
            chunk = page_text[start:end].strip()

            if chunk:
                chunks.append(chunk)
                metadata.append({
                    "page": page["page"],
                    "source": f"Page {page['page']}"
                })

            if end >= len(page_text):
                break

            start = max(end - overlap, start + 1)

    return chunks, metadata

## 8. Build the FAISS Knowledge Base

Embeddings are normalized and searched using inner-product similarity, equivalent to cosine similarity for normalized vectors.

In [7]:
def build_vector_store(chunks, metadata):
    if not chunks:
        raise ValueError("No document chunks available.")

    embeddings = embedding_model.encode(
        chunks,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    STATE["chunks"] = chunks
    STATE["chunk_metadata"] = metadata
    STATE["embeddings"] = embeddings
    STATE["faiss_index"] = index

    return index


def process_document(file_path):
    raw_text, pages = extract_text_from_file(file_path)
    chunks, metadata = chunk_pages(pages)
    build_vector_store(chunks, metadata)

    STATE["document_name"] = Path(file_path).name
    STATE["raw_text"] = raw_text
    STATE["chat_history"] = []
    STATE["last_answer"] = ""
    STATE["last_summary"] = ""
    STATE["last_quiz"] = ""

    return (
        f"### ✅ Document processed\n\n"
        f"**File:** {STATE['document_name']}  \n"
        f"**Characters:** {len(raw_text):,}  \n"
        f"**Chunks:** {len(chunks)}  \n"
        f"**Embedding dimension:** {STATE['embeddings'].shape[1]}"
    )

## 9. Semantic Retrieval

In [8]:
def retrieve_context(query, top_k=4):
    if STATE["faiss_index"] is None:
        raise ValueError("Please upload and process a document first.")

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = STATE["faiss_index"].search(
        query_embedding,
        min(top_k, len(STATE["chunks"]))
    )

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append({
            "chunk": STATE["chunks"][idx],
            "source": STATE["chunk_metadata"][idx]["source"],
            "score": float(score)
        })
    return results


def format_retrieved_context(results):
    parts = []
    for i, r in enumerate(results, start=1):
        parts.append(
            f"[Source {i} — {r['source']} — similarity {r['score']:.3f}]\n{r['chunk']}"
        )
    return "\n\n".join(parts)

## 10. Gemini RAG Response Generator

The LLM is instructed to answer only from retrieved study material and to acknowledge missing information.

In [17]:
def call_gemini(prompt):
    if client is None:
        raise RuntimeError(
            "Gemini API key is not configured. Rerun the API configuration cell."
        )

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    return response.text.strip()


def answer_question(question, top_k=4):
    if not question or not question.strip():
        raise ValueError("Please enter a question.")

    results = retrieve_context(question, top_k)
    context = format_retrieved_context(results)

    prompt = (
        "You are AI StudyMate, an academic study assistant.\n\n"
        "Answer the student's question using ONLY the retrieved study material below.\n"
        "Rules:\n"
        "1. Ground the answer in the supplied context.\n"
        "2. Do not invent facts not supported by the context.\n"
        "3. If the context is insufficient, clearly say so.\n"
        "4. Explain at a university-student level.\n"
        "5. Use concise paragraphs and bullets where useful.\n\n"
        "STUDENT QUESTION:\n" + question + "\n\n"
        "RETRIEVED STUDY MATERIAL:\n" + context
    )

    answer = call_gemini(prompt)

    STATE["last_answer"] = answer
    STATE["chat_history"].append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "question": question,
        "answer": answer,
        "sources": [{"source": r["source"], "score": r["score"]} for r in results]
    })
    save_history()

    return answer, results

## 11. Summary Generator

In [18]:
def generate_summary():
    if not STATE["chunks"]:
        raise ValueError("Please upload and process a document first.")

    max_chunks = min(len(STATE["chunks"]), 25)
    context = "\n\n".join(
        f"[{STATE['chunk_metadata'][i]['source']}] {STATE['chunks'][i]}"
        for i in range(max_chunks)
    )

    prompt = (
        "You are an academic study assistant.\n\n"
        "Create a structured exam-preparation summary from the following material.\n"
        "Include major concepts, brief explanations, important definitions, and "
        "technical terms or formulas when present. Do not invent information.\n\n"
        "STUDY MATERIAL:\n" + context
    )

    summary = call_gemini(prompt)
    STATE["last_summary"] = summary
    save_history()
    return summary

## 12. Quiz Generator

In [19]:
def generate_quiz(num_questions=5):
    if not STATE["chunks"]:
        raise ValueError("Please upload and process a document first.")

    max_chunks = min(len(STATE["chunks"]), 20)
    context = "\n\n".join(STATE["chunks"][:max_chunks])

    prompt = (
        f"You are an academic quiz generator. Create {num_questions} multiple-choice "
        "questions from the supplied study material. For each question provide four "
        "options A-D, the correct answer, and a one-sentence explanation. "
        "Use only the material and avoid ambiguous questions.\n\n"
        "STUDY MATERIAL:\n" + context
    )

    quiz = call_gemini(prompt)
    STATE["last_quiz"] = quiz
    save_history()
    return quiz

## 13. Storage

In [20]:
def save_history():
    payload = {
        "document_name": STATE["document_name"],
        "updated_at": datetime.now().isoformat(timespec="seconds"),
        "chat_history": STATE["chat_history"],
        "last_summary": STATE["last_summary"],
        "last_quiz": STATE["last_quiz"],
    }
    HISTORY_FILE.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )


def load_history():
    if not HISTORY_FILE.exists():
        return {"message": "No saved study history found."}
    return json.loads(HISTORY_FILE.read_text(encoding="utf-8"))

## 14. Gradio Helper Functions

In [21]:
def make_sources_markdown(results):
    if not results:
        return "No sources found."

    lines = ["### 📚 Retrieved Sources"]
    for i, result in enumerate(results, start=1):
        preview = result["chunk"].replace("\n", " ")
        if len(preview) > 180:
            preview = preview[:180] + "..."
        lines.append(
            f"{i}. **{result['source']}** — similarity `{result['score']:.3f}`  \n"
            f"   {preview}"
        )
    return "\n".join(lines)


def save_text_file(content, filename):
    path = Path(filename)
    path.write_text(content or "", encoding="utf-8")
    return str(path)


def ui_process_document(file_path):
    try:
        return process_document(file_path)
    except Exception as e:
        return f"❌ Error: {e}"


def ui_ask(question):
    try:
        answer, results = answer_question(question)
        return answer, make_sources_markdown(results)
    except Exception as e:
        return f"❌ Error: {e}", ""


def ui_summary():
    try:
        return generate_summary()
    except Exception as e:
        return f"❌ Error: {e}"


def ui_quiz(num_questions):
    try:
        return generate_quiz(int(num_questions))
    except Exception as e:
        return f"❌ Error: {e}"


def ui_history():
    try:
        return json.dumps(load_history(), indent=2, ensure_ascii=False)
    except Exception as e:
        return f"❌ Error: {e}"


def ui_download_answer():
    return save_text_file(STATE["last_answer"], "studymate_answer.txt") if STATE["last_answer"] else None


def ui_download_summary():
    return save_text_file(STATE["last_summary"], "studymate_summary.md") if STATE["last_summary"] else None


def ui_download_quiz():
    return save_text_file(STATE["last_quiz"], "studymate_quiz.md") if STATE["last_quiz"] else None

## 15. Gradio Interface

This maps directly to the required project pipeline:

- **Input:** upload document and enter a question
- **AI Logic:** embeddings, FAISS retrieval, RAG, summary, quiz
- **Interface:** Gradio
- **Storage:** JSON study history
- **Action:** answer, summarize, quiz, save/download

In [22]:
with gr.Blocks(title="AI StudyMate", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 🤖 AI StudyMate\n"
        "### AI-Powered Student Study Assistant\n\n"
        "**Pipeline:** Input → AI Logic → Interface → Storage → Action"
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## 📚 1. Upload Study Material")
            file_input = gr.File(
                label="Upload PDF or TXT",
                file_types=[".pdf", ".txt"],
                type="filepath"
            )
            process_btn = gr.Button("📖 Process Document", variant="primary")
            document_status = gr.Markdown("No document processed yet.")

        with gr.Column(scale=2):
            gr.Markdown("## 🔍 2. Ask Your Study Assistant")
            question_input = gr.Textbox(
                label="Question",
                placeholder="e.g. What is CPU scheduling?",
                lines=3
            )
            ask_btn = gr.Button("🤖 Ask AI", variant="primary")

    with gr.Row():
        with gr.Column():
            answer_output = gr.Markdown()
            download_answer_btn = gr.Button("⬇️ Prepare Answer Download")
            answer_file = gr.File(label="Answer File", interactive=False)
        with gr.Column():
            sources_output = gr.Markdown()

    gr.Markdown("---")

    with gr.Row():
        with gr.Column():
            gr.Markdown("## 📝 3. Generate Summary")
            summary_btn = gr.Button("Generate Summary")
            summary_output = gr.Markdown()
            download_summary_btn = gr.Button("⬇️ Prepare Summary Download")
            summary_file = gr.File(label="Summary File", interactive=False)

        with gr.Column():
            gr.Markdown("## ❓ 4. Generate Quiz")
            num_questions = gr.Slider(
                minimum=3, maximum=10, value=5, step=1,
                label="Number of Questions"
            )
            quiz_btn = gr.Button("Generate Quiz")
            quiz_output = gr.Markdown()
            download_quiz_btn = gr.Button("⬇️ Prepare Quiz Download")
            quiz_file = gr.File(label="Quiz File", interactive=False)

    gr.Markdown("---")
    gr.Markdown("## 💾 5. Stored Study Session")
    history_btn = gr.Button("View Saved History")
    history_output = gr.Code(language="json", label="study_history.json")

    process_btn.click(ui_process_document, file_input, document_status)
    ask_btn.click(ui_ask, question_input, [answer_output, sources_output])
    summary_btn.click(ui_summary, None, summary_output)
    quiz_btn.click(ui_quiz, num_questions, quiz_output)
    history_btn.click(ui_history, None, history_output)

    download_answer_btn.click(ui_download_answer, None, answer_file)
    download_summary_btn.click(ui_download_summary, None, summary_file)
    download_quiz_btn.click(ui_download_quiz, None, quiz_file)

print("Gradio application constructed successfully.")

/tmp/ipykernel_8084/1195487837.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="AI StudyMate", theme=gr.themes.Soft()) as demo:


Gradio application constructed successfully.


## 16. Launch the Application

Run this cell last. Gradio will provide a shareable URL in Colab.

In [23]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://39f21cea7a1536cd07.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://39f21cea7a1536cd07.gradio.live


# 17. Testing Checklist

### Input
- Upload a PDF.
- Process it.
- Confirm chunks are created.
- Ask a question.

### AI Logic
- Confirm relevant source chunks appear.
- Confirm the answer is grounded in the uploaded material.
- Generate a summary.
- Generate a quiz.

### Storage
- Ask two questions.
- View saved history.
- Confirm `study_history.json` is created.

### Actions
- Generate summary.
- Generate quiz.
- Prepare answer download.
- Prepare summary download.
- Prepare quiz download.

### Example questions
- What is the main concept discussed in this document?
- Explain CPU scheduling.
- What is the difference between a process and a program?
- Summarize the most important points for an exam.

# 18. AI Project Pipeline — Final Documentation

## INPUT
PDF/TXT study material and student questions.

## AI LOGIC
1. Text extraction with PyPDF.
2. Text cleaning and chunking.
3. Sentence Transformer embeddings.
4. FAISS vector search.
5. Retrieval of relevant chunks.
6. Retrieval-Augmented Generation with Gemini.
7. Grounded answers.
8. Summary and quiz generation.

## INTERFACE
Gradio web interface for upload, questions, answers, sources, summaries, quizzes, history, and downloads.

## STORAGE
Session state plus persistent `study_history.json`.

## ACTION
Answer questions, generate summaries, create quizzes, save study sessions, and prepare downloadable outputs.

## COMPLETE PIPELINE
```text
Student
   ↓
PDF / TXT / Question
   ↓
Text Extraction
   ↓
Chunking
   ↓
Sentence Transformer
   ↓
FAISS Vector Search
   ↓
Relevant Context
   ↓
Gemini RAG
   ↓
Answer / Summary / Quiz
   ↓
Gradio Interface
   ↓
JSON Storage
   ↓
Download / Study Actions
```

# 19. Conclusion

AI StudyMate demonstrates a complete AI project pipeline in a compact Python/Colab environment. It combines document processing, natural language processing, semantic embeddings, vector search, retrieval-augmented generation, a web interface, lightweight storage, and useful user-facing actions.

The architecture deliberately separates Input, AI Logic, Interface, Storage, and Action so the project can be clearly explained during the laboratory demonstration.